In [1]:
import os
import glob
import subprocess
import re
from concurrent.futures import ThreadPoolExecutor

# ================= CONFIGURATION =================
# 你的原始 ADNI DICOM 根目录
INPUT_DIR = "../autodl-tmp/ADNI_raw_data"  
# 转换后的 NIfTI 文件保存目录
OUTPUT_DIR = "../autodl-tmp/NIfTI_output"  
# 线程数（根据你的服务器 CPU 核心数调整，ADNI 数据量大，多线程能快很多）
NUM_THREADS = 8  
# ==========================================

def process_dicom_folder(dcm_folder):
    """
    处理单个包含 .dcm 的最底层文件夹
    """
    # 获取该文件夹下所有的 dcm 文件
    dcm_files = glob.glob(os.path.join(dcm_folder, "*.dcm"))
    if not dcm_files:
        return
    
    # 随便取第一个文件来解析名称
    sample_file = os.path.basename(dcm_files[0])
    
    # 使用正则表达式提取 Subject ID (例如: 010_S_0420) 和 Image ID (例如: I193085)
    # ADNI 的 Subject ID 格式通常为: 3位数字_S_4位数字
    subject_match = re.search(r'\d{3}_S_\d{4}', sample_file)
    # ADNI 的 Image ID 格式通常为: I + 多位数字
    image_id_match = re.search(r'I\d{4,8}', sample_file)
    
    if subject_match and image_id_match:
        subject_id = subject_match.group()
        image_id = image_id_match.group()
        
        # 构造输出文件名: SubjectID_ImageID (不带后缀，dcm2niix会自动加 .nii.gz)
        # 例如: 010_S_0420_I193085
        output_filename = f"{subject_id}_{image_id}"
        
        # 构造 dcm2niix 命令
        # -z y: 压缩为 .nii.gz
        # -f: 指定输出文件名
        # -o: 输出目录
        cmd =[
            "dcm2niix",
            "-z", "y",
            "-f", output_filename,
            "-o", OUTPUT_DIR,
            dcm_folder
        ]
        
        try:
            # 执行命令，隐藏原本庞大的控制台输出
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            print(f"[成功] {subject_id} - {image_id} 转换完成")
        except subprocess.CalledProcessError:
            print(f"[错误] dcm2niix 转换失败: {dcm_folder}")
    else:
        print(f"[跳过] 无法从文件名解析 ID: {sample_file}")

def main():
    # 确保输出目录存在
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print("正在扫描目录，寻找包含 .dcm 的最底层文件夹...")
    dcm_folders = set()
    
    # os.walk 递归遍历所有目录
    for root, dirs, files in os.walk(INPUT_DIR):
        for file in files:
            if file.endswith(".dcm"):
                dcm_folders.add(root)
                break # 找到一个dcm就说明这个文件夹需要处理，跳出当前文件的循环
                
    dcm_folders = list(dcm_folders)
    print(f"共发现 {len(dcm_folders)} 个待处理的扫描序列文件夹。")
    print(f"开始使用 {NUM_THREADS} 个线程进行批量转换...\n")
    
    # 使用线程池并发执行，大幅缩短时间
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        executor.map(process_dicom_folder, dcm_folders)
        
    print("\n全部转换任务执行完毕！")

if __name__ == "__main__":
    main()

正在扫描目录，寻找包含 .dcm 的最底层文件夹...
共发现 531 个待处理的扫描序列文件夹。
开始使用 8 个线程进行批量转换...

[成功] 099_S_0534 - I55083 转换完成
[成功] 141_S_1004 - I105124 转换完成
[成功] 023_S_1247 - I41203 转换完成
[成功] 098_S_0667 - I189353 转换完成
[成功] 052_S_1168 - I32350 转换完成
[成功] 032_S_1101 - I37523 转换完成
[成功] 033_S_0906 - I25052 转换完成
[成功] 037_S_0467 - I106120 转换完成
[成功] 137_S_0994 - I159945 转换完成
[成功] 130_S_0285 - I113594 转换完成
[成功] 037_S_1225 - I69231 转换完成
[成功] 136_S_0195 - I74946 转换完成
[成功] 123_S_0072 - I9752 转换完成
[成功] 128_S_0947 - I103801 转换完成
[成功] 037_S_0150 - I24775 转换完成
[成功] 029_S_0843 - I82864 转换完成
[成功] 129_S_0778 - I155001 转换完成
[成功] 023_S_0081 - I19650 转换完成
[成功] 007_S_1222 - I166407 转换完成
[成功] 005_S_1224 - I137795 转换完成
[成功] 013_S_1276 - I141081 转换完成
[成功] 007_S_0249 - I48166 转换完成
[成功] 037_S_0539 - I62156 转换完成
[成功] 033_S_1086 - I33522 转换完成
[成功] 041_S_0549 - I48337 转换完成
[成功] 033_S_0920 - I52202 转换完成
[成功] 013_S_0860 - I24924 转换完成
[成功] 029_S_0914 - I165872 转换完成
[成功] 036_S_0945 - I27442 转换完成
[成功] 033_S_1086 - I129094 转换完成
[成功] 057_S_1007 - 

In [2]:
# import os
# import subprocess
# import re
# from concurrent.futures import ThreadPoolExecutor

# # ================= 配置区 =================
# # 你的原始 ADNI DICOM 根目录
# # 【注意】请确认你的解压路径，有时候解压后会多一层，变成 /autodl-tmp/ADNI_raw_data/ADNI
# INPUT_DIR = "../autodl-tmp/ADNI_raw_data"  
# OUTPUT_DIR = "../autodl-tmp/NIfTI_output"  
# NUM_THREADS = 8  
# # ==========================================

# def process_dicom_folder(dcm_folder):
#     """
#     处理单个最底层文件夹
#     """
#     # 获取文件夹下的所有文件（排除子文件夹）
#     files_in_dir =[f for f in os.listdir(dcm_folder) if os.path.isfile(os.path.join(dcm_folder, f))]
    
#     # 过滤掉常见的 ADNI 附属说明文件，剩下的通常就是 DICOM 图像本身（不管它有没有后缀）
#     valid_files =[f for f in files_in_dir if not f.lower().endswith(('.xml', '.csv', '.txt', '.json', '.pdf'))]
    
#     if not valid_files:
#         return
    
#     # 随便取第一个有效文件的【完整绝对路径】来解析 ID
#     sample_file_path = os.path.join(dcm_folder, valid_files[0])
    
#     # 【核心升级】在“完整路径”中寻找 Subject ID 和 Image ID
#     # 哪怕文件名叫 1-01.DCM，只要它的父文件夹叫 I193085，就能精准抓取！
#     subject_match = re.search(r'\d{3}_S_\d{4}', sample_file_path)
#     image_id_match = re.search(r'I\d{4,8}', sample_file_path)
    
#     if subject_match and image_id_match:
#         subject_id = subject_match.group()
#         image_id = image_id_match.group()
        
#         output_filename = f"{subject_id}_{image_id}"
        
#         cmd =[
#             "dcm2niix",
#             "-z", "y",
#             "-f", output_filename,
#             "-o", OUTPUT_DIR,
#             dcm_folder  # 直接把文件夹路径丢给 dcm2niix，它极其聪明，能自动读取里面的所有切片
#         ]
        
#         try:
#             # 运行 dcm2niix
#             subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
#             print(f"[成功] {subject_id} - {image_id} 转换完成")
#         except subprocess.CalledProcessError:
#             print(f"[错误] dcm2niix 转换失败: {dcm_folder}")
#     else:
#         # 如果真的找不到 ID，打印出来看看到底是什么奇怪的路径
#         pass 
#         # print(f"[跳过] 路径中缺少受试者ID或图像ID: {sample_file_path}")

# def main():
#     os.makedirs(OUTPUT_DIR, exist_ok=True)
    
#     print("正在扫描目录，寻找最底层数据文件夹...")
#     dcm_folders = set()
    
#     for root, dirs, files in os.walk(INPUT_DIR):
#         # 只要当前目录下有文件（排除全是文件夹的中间层）
#         if files:
#             # 过滤掉元数据文件，看看还有没有剩下的实体数据
#             data_files =[f for f in files if not f.lower().endswith(('.xml', '.csv', '.txt', '.json', '.pdf'))]
#             if data_files:
#                 dcm_folders.add(root)
                
#     dcm_folders = list(dcm_folders)
#     print(f"共发现 {len(dcm_folders)} 个待处理的扫描序列文件夹。")
    
#     if len(dcm_folders) == 0:
#         print("【警告】还是 0 个？请检查：")
#         print("1. /autodl-tmp/ADNI_raw_data 路径是否写错？（试试看进入目录用 ls 命令看一眼）")
#         print("2. 是否解压后多了一层目录？比如 /autodl-tmp/ADNI_raw_data/ADNI")
#         return

#     print(f"开始使用 {NUM_THREADS} 个线程进行批量转换...\n")
    
#     with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
#         executor.map(process_dicom_folder, dcm_folders)
        
#     print("\n全部转换任务执行完毕！")

# if __name__ == "__main__":
#     main()